In [11]:
from itertools import combinations

In [12]:
# Sample dataset
transactions = [
    ['milk', 'bread', 'butter'],
    ['bread', 'butter'],
    ['milk', 'bread'],
    ['milk', 'bread', 'butter', 'jam'],
    ['bread', 'jam']
]

In [13]:
# Function to calculate support
def get_support(itemset, transactions):
    count = 0
    for transaction in transactions:
        if set(itemset).issubset(set(transaction)):
            count += 1
    return count / len(transactions)


In [14]:
# Function to generate candidate itemsets
def generate_candidates(prev_freq_items, k):
    candidates = []
    length = len(prev_freq_items)
    for i in range(length):
        for j in range(i + 1, length):
            l1 = list(prev_freq_items[i])
            l2 = list(prev_freq_items[j])
            l1.sort()
            l2.sort()
            if l1[:k - 2] == l2[:k - 2]:
                candidate = tuple(sorted(set(prev_freq_items[i]) | set(prev_freq_items[j])))
                if candidate not in candidates:
                    candidates.append(candidate)
    return candidates

In [15]:
# Apriori algorithm
def apriori(transactions, min_support):
    itemsets = []
    support_data = {}

    # Step 1: Create 1-itemsets
    items = set(item for transaction in transactions for item in transaction)
    candidates = [(item,) for item in items]

    # Step 2: Filter 1-itemsets by support
    freq_items = []
    for item in candidates:
        support = get_support(item, transactions)
        if support >= min_support:
            freq_items.append(item)
            support_data[item] = support

    itemsets.extend(freq_items)
    k = 2

    while freq_items:
        candidates = generate_candidates(freq_items, k)
        freq_items = []
        for item in candidates:
            support = get_support(item, transactions)
            if support >= min_support:
                freq_items.append(item)
                support_data[item] = support
        itemsets.extend(freq_items)
        k += 1

    return itemsets, support_data

In [16]:
# Function to generate association rules
def generate_rules(itemsets, support_data, min_confidence):
    rules = []
    for itemset in itemsets:
        if len(itemset) >= 2:
            for i in range(1, len(itemset)):
                for antecedent in combinations(itemset, i):
                    consequent = tuple(set(itemset) - set(antecedent))
                    if consequent:
                        conf = support_data[itemset] / support_data[antecedent]
                        if conf >= min_confidence:
                            rules.append((antecedent, consequent, conf))
    return rules

In [17]:
# Run Apriori
min_support = 0.5
min_confidence = 0.7

frequent_itemsets, support_data = apriori(transactions, min_support)
association_rules = generate_rules(frequent_itemsets, support_data, min_confidence)

In [18]:
# Output results
print("Frequent Itemsets:")
for item in frequent_itemsets:
    print(f"{item}: {support_data[item]:.2f}")

print("\nAssociation Rules:")
for rule in association_rules:
    print(f"{rule[0]} => {rule[1]} (confidence: {rule[2]:.2f})")

Frequent Itemsets:
('milk',): 0.60
('bread',): 1.00
('butter',): 0.60
('bread', 'milk'): 0.60
('bread', 'butter'): 0.60

Association Rules:
('milk',) => ('bread',) (confidence: 1.00)
('butter',) => ('bread',) (confidence: 1.00)
